# Adım 3 — Spark Structured Streaming + Delta LakeBu notebook, **Bronze → Silver → Gold** üç katmanlı Delta Lake mimarisini belgeler ve her katmandaki verileri inceler.## Katman Mimarisi (Medallion Architecture)```  Kafka Stream       │       ▼  ┌──────────┐   Ham JSON, şemasız, append-only  │  BRONZE  │   Partitioning: ingestion_year/month/day  └────┬─────┘       │ readStream → from_json + temizlik       ▼  ┌──────────┐   Şema uygulanmış, null/dup/inf temizlenmiş  │  SILVER  │   ~60 kolon, tip dönüşümü yapılmış  └────┬─────┘       │ readStream → FeatureEngineer       ▼  ┌──────────┐   ML-ready: 5 yeni türetilmiş özellik  │   GOLD   │   + StringIndexer label + VectorAssembler  └──────────┘```

In [ ]:
# Spark oturumu ve Delta Lake tablolarını yükleimport os, sysfrom pathlib import Path# Proje köküPROJECT_ROOT = Path(os.pardir).resolve()DELTA_ROOT = PROJECT_ROOT / "delta-storage"# Katman yollarılayers = {    "Bronze": DELTA_ROOT / "bronze" / "network_traffic",    "Silver": DELTA_ROOT / "silver" / "network_traffic",    "Gold":   DELTA_ROOT / "gold" / "ml_ready",}print("Delta Lake Katman Durumu")print("═" * 60)for name, path in layers.items():    exists = path.exists()    if exists:        parquets = list(path.rglob("*.parquet"))        size_mb = sum(p.stat().st_size for p in parquets) / (1024*1024)        print(f"  {name:<8} ✅ {len(parquets)} parquet dosyası, {size_mb:.1f} MB")    else:        print(f"  {name:<8} ❌ Henüz oluşturulmamış")

## Bronze Katmanı**Görev:** Kafka'dan gelen ham JSON mesajları olduğu gibi depolar.| Özellik | Değer ||---------|-------|| Format | Delta Lake (append-only) || Şema | `json_payload` (string) + `ingestion_time` (timestamp) || Partitioning | `ingestion_year`, `ingestion_month`, `ingestion_day` || Trigger | Her 30 saniye || Kaynak script | `spark/streaming/bronze_layer.py` |

In [ ]:
# Bronze tablosundan örnek veriimport pandas as pdbronze_path = DELTA_ROOT / "bronze" / "network_traffic"if bronze_path.exists():    try:        parquets = sorted(bronze_path.rglob("*.parquet"))[:2]        if parquets:            df = pd.read_parquet(parquets[0])            print(f"Bronze — Kolon sayısı: {len(df.columns)}, Satır (bu parquet): {len(df)}")            print(f"Kolonlar: {list(df.columns)}")            print(f"\nÖrnek kayıt:")            print(df.head(2).to_string())    except Exception as e:        print(f"Okuma hatası: {e}")else:    print("Bronze tablosu henüz oluşturulmamış — streaming pipeline çalıştırın.")

## Silver Katmanı**Görev:** Bronze'dan okur, JSON'u parse eder, şema uygular, temizlik yapar.**Temizlik adımları:**1. `from_json()` ile JSON → yapılandırılmış kolonlar2. Kolon isimlerindeki `.` → `_` ile değiştirilir3. ARP ve ICMP kolonları (8 adet) kaldırılır4. Sayısal kolonlardaki `inf`/`-inf` → `null` → `0.0`5. String eksik değerler → `"unknown"`6. `flow_id` bazlı duplikat kaldırma7. Port kolonları integer'a cast edilir

In [ ]:
# Silver tablosundan örnek verisilver_path = DELTA_ROOT / "silver" / "network_traffic"if silver_path.exists():    try:        parquets = sorted(silver_path.rglob("*.parquet"))[:2]        if parquets:            df = pd.read_parquet(parquets[0])            print(f"Silver — Kolon sayısı: {len(df.columns)}, Satır (bu parquet): {len(df)}")            print(f"\nKolon listesi:")            for i, col in enumerate(df.columns):                print(f"  {i+1:2}. {col:<30} {str(df[col].dtype)}")            print(f"\nÖrnek kayıt (ilk 5 kolon):")            print(df.iloc[:3, :5].to_string())    except Exception as e:        print(f"Okuma hatası: {e}")else:    print("Silver tablosu henüz oluşturulmamış.")

## Gold Katmanı**Görev:** Silver'dan okur, 5 yeni feature üretir, ML'ye hazır hale getirir.**Türetilen 5 özellik:**| # | Feature | Formül | Tespit Ettiği Saldırı ||---|---------|--------|----------------------|| 1 | `traffic_asymmetry_ratio` | tcp_ack / (tcp_seq + 1) | DDoS Flood, SYN Flood || 2 | `pkt_size_cv` | abs(tcp_len − tcp_payload) / (tcp_len + 1) | Port Scanning || 3 | `flow_intensity` | tcp_len / (frame_time_relative + ε) | DDoS, Brute-Force || 4 | `iat_regularity` | std(arrival_intervals) / (mean + ε) | Botnet, MITM || 5 | `conn_efficiency` | tcp_payload / (tcp_len + 1) | Reconnaissance, ICMP Flood |

In [ ]:
# Gold tablosundan örnek veri ve türetilmiş feature'ları göstergold_paths = [    DELTA_ROOT / "gold" / "ml_ready_compact",    DELTA_ROOT / "gold" / "ml_ready",]gold_df = Nonefor gp in gold_paths:    if gp.exists():        try:            parquets = sorted(gp.rglob("*.parquet"))            if parquets:                gold_df = pd.concat([pd.read_parquet(p) for p in parquets[:3]], ignore_index=True)                print(f"Gold — Kolon sayısı: {len(gold_df.columns)}, Satır (örneklem): {len(gold_df)}")                break        except Exception as e:            print(f"Okuma hatası ({gp.name}): {e}")if gold_df is not None:    eng_features = ["traffic_asymmetry_ratio", "pkt_size_cv", "flow_intensity", "iat_regularity", "conn_efficiency"]    found = [f for f in eng_features if f in gold_df.columns]    print(f"\nTüretilmiş özellikler ({len(found)}/5 bulundu): {found}")    if found:        print(f"\nİstatistikler:")        print(gold_df[found].describe().round(4).to_string())else:    print("Gold tablosu henüz oluşturulmamış.")

## Pipeline Çalıştırma```bash# Tam pipeline (Bronze → Silver → Gold)docker exec -it spark-master spark-submit \  --packages io.delta:delta-core_2.12:2.4.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.2 \  /opt/bitnami/spark/spark/run_streaming_pipeline.py```Pipeline başarıyla başladığında:```[1/3] Starting Bronze Layer Ingestion...[2/3] Starting Silver Layer (Cleaning & Parsing)...[3/3] Starting Gold Layer (Feature Engineering)...✅ Tüm streaming katmanları (Bronze -> Silver -> Gold) başlatıldı.```